# **Data Preprocessing**

In this section, we will perform **data preprocessing** on the **Customer Churn Dataset** to prepare it for model training. Data preprocessing is an essential step in any machine learning project, as it ensures that the data is clean, properly formatted, and ready for analysis.

### **Goals of Data Preprocessing**:
1. **Handle Missing Values**: Identify and impute or remove missing values from the dataset to prevent any bias or errors in the model.
2. **Categorical Feature Encoding**: Convert categorical variables (such as `Gender`, `Contract`, `Payment Method`, etc.) into numerical values using techniques like **One-Hot Encoding** or **Label Encoding**.
3. **Feature Scaling/Normalization**: Scale numerical features to a standard range or distribution to improve model performance, especially for algorithms like KNN and SVM.
4. **Outlier Treatment**: Address any identified outliers from the EDA phase (if necessary).
5. **Imbalanced Classes**: If churn data is imbalanced (i.e., there are more non-churn customers than churn customers), apply techniques such as **SMOTE** or **undersampling** to balance the dataset.

### **Steps in Data Preprocessing**:
1. **Handling Missing Data**:
   - Identify columns with missing data and choose whether to **impute** missing values or **remove** rows/columns.
   
2. **Outlier Treatment**:
   - Based on the outlier analysis, handle the outliers appropriately (e.g., removing or capping them).

3. **Feature Selection**:
   - Based on the EDA insights, select the most relevant features for model training to improve performance and reduce overfitting.

4. **Encoding Categorical Variables**:
   - Apply **One-Hot Encoding** or **Label Encoding** for categorical columns to convert them into numeric values.
   
5. **Feature Scaling**:
   - Normalize or standardize the numerical features to ensure they are on the same scale.
   
6. **Class Imbalance**:
   - If the target variable is imbalanced, consider applying resampling techniques to balance the classes.

Once the data is preprocessed, we will be ready to move on to building predictive models.


# Load Data

In [19]:
# Import Libraries

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load Data

df = pd.read_excel('Telco_customer_churn.xlsx')

### **Step 1: Handle Missing Data & Duplicates**

#### Fixing Missing Data Across All Columns

To handle any **empty strings** or **non-visible characters** in the entire dataset, we will:
- **Replace empty strings or spaces with NaN** for all columns.


In [20]:
# Replace empty strings or spaces with NaN across all columns
df.replace(" ", pd.NA, inplace=True)


In [21]:
missing_columns = df.columns[df.isnull().any()]
missing_columns


Index(['Total Charges', 'Churn Reason'], dtype='object')

##### **Handling Missing Data in "Churn Reason" & "Total Charges"**

The **"Churn Reason"** column contains missing values, but these are expected for customers who did not churn. Since the missing values indicate that these customers did not leave, we will handle them as **NaN** for the non-churned customers, as they do not have a reason for churn.

Since **"Total Charges"** has a strong correlation with "Monthly Charges" (0.65), we can fill the missing values in the "Total Charges" column by using the relationship between **"Monthly Charges"** and **"Tenure Months"**.


##### **What to Do**:
- We will replace the missing values in the **"Churn Reason"** column with **NaN**, indicating the absence of churn reason for non-churned customers.

- For rows with missing values in **"Total Charges"**, we will fill them by multiplying the **"Monthly Charges"** by the **"Tenure Months"**.


In [22]:
# Fill missing values in "Churn Reason" with NaN for non-churned customers
df['Churn Reason'] = df['Churn Reason'].fillna('no_reason_given')

# Verify that the missing values in 'Churn Reason' are filled
df['Churn Reason'].isnull().sum()


np.int64(0)

In [23]:
# Fill missing Total Charges using the formula: Monthly Charges * Tenure Months
df['Total Charges'] = df['Total Charges'].fillna(df['Monthly Charges'] * df['Tenure Months'])

# Verify the result by checking for missing values again
df['Total Charges'].isnull().sum()


np.int64(0)

In [24]:
df.duplicated().sum()

np.int64(0)

**No duplicates found in the dataset.**

### **Step 2: Outlier Treatment**

Based on the outlier analysis from the EDA phase, we'll apply Robust Scaling to the numerical features to mitigate the impact of outliers on our models. Robust Scaling uses the median and interquartile range, making it less sensitive to outliers compared to standard scaling methods.

Only the **"Total Charges"** and **"CLTV"** columns will be scaled using RobustScaler, as they contain significant outliers that could affect model performance. The other numerical features will be left unchanged, as they do not exhibit significant outliers.

In [25]:
from sklearn.preprocessing import RobustScaler

robust_scaler = RobustScaler()
df[['Total Charges', 'CLTV']] = robust_scaler.fit_transform(df[['Total Charges', 'CLTV']])

### **Step 3: Feature Selection**

In this step, we will perform an **initial feature selection** by removing irrelevant columns that do not contribute to the prediction model. Based on the EDA insights, we will drop the following columns:

- CustomerID
- Count
- Country
- State
- City
- Zip Code
- Lat Long
- Latitude
- Longitude
- Churn Value
- Churn Reason (since it is only relevant for churned customers and we'll use it later for the Business Insights)

In [26]:
# Initial feature selection: Dropping irrelevant and constant columns
df.drop(columns=['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Churn Value', 'Churn Reason'], inplace=True)

# Verify the columns after initial feature selection
df.shape


(7043, 22)

Initially we have 33 columns, after dropping the irrelevant ones, we will have 22 columns left for model training.

### **Step 4: Encoding Categorical Variables**

Now that we have performed initial feature selection, we will encode the categorical variables into numerical format so they can be used by machine learning models.

#### **What to Do**:

1. **Label Encoding**: Apply this to binary categorical columns, such as:
   - `Gender`, `Senior Citizen`, `Partner`, `Dependents`, `Churn Label`..........

2. **Fix Data Types**: Convert any incorrectly labeled `object` columns (such as `Total Charges`) to the appropriate numerical data types (`float64` or `int64`).


#### Data type of all columns


In [27]:
# Check the data types of all columns in the dataset
df.dtypes

# List of columns based on data types
categorical_columns = df.select_dtypes(include=['object']).columns  # string columns
numerical_columns = df.select_dtypes(include=['float64', 'int64']).columns  # numeric columns

# Display the columns to confirm
print("Categorical Columns: ", categorical_columns)
print("Numerical Columns: ", numerical_columns)


Categorical Columns:  Index(['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service',
       'Multiple Lines', 'Internet Service', 'Online Security',
       'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
       'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method',
       'Churn Label'],
      dtype='object')
Numerical Columns:  Index(['Tenure Months', 'Monthly Charges', 'Total Charges', 'Churn Score',
       'CLTV'],
      dtype='object')


#### A) **Fixing Data Types for All Numerical Columns**

In [28]:
# Convert all relevant numerical columns to the appropriate data types
df['Tenure Months'] = df['Tenure Months'].astype('int64')  # 'Tenure Months' should be integer
df['Monthly Charges'] = df['Monthly Charges'].astype('float64')  # 'Monthly Charges' should be float
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')  # 'Total Charges' should be float
df['Churn Score'] = df['Churn Score'].astype('int64')  # 'Churn Score' should be integer
df['CLTV'] = df['CLTV'].astype('float64')  # 'CLTV' should be float


#### B) **Encoding Categorical Variables**

In this step, we will apply the following encoding strategies:
1. **Label Encoding**: We will use Label Encoding for categorical features (like `Contract`, `Payment Method`, etc.) to convert them into numeric labels.

This approach ensures:
- No information loss
- Proper handling of text data
- Model-ready numerical features

In [29]:
from sklearn.preprocessing import LabelEncoder

# List of columns to apply Label Encoding
label_encode_columns = [
    'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 
    'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 
    'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 
    'Contract', 'Paperless Billing', 'Payment Method', 'Churn Label'
]

# Initialize LabelEncoder
le = LabelEncoder()

# Apply Label Encoding to the listed columns
for col in label_encode_columns:
    df[col] = le.fit_transform(df[col])



Now all the categorical variables have been encoded, and the numerical columns have been converted to the appropriate data types, making our dataset ready for model training.

### **Step 6: Feature Scaling**

In this step, we will apply **feature scaling** to the numerical columns in the dataset. Scaling is important for models that are sensitive to the magnitude of the features (e.g., **KNN**, **Logistic Regression**, **SVM**).

#### **What to Do**:
1. **Standard Scaling**: We will scale the numerical features to have a **mean of 0** and a **standard deviation of 1** using **StandardScaler**.
2. **Apply Scaling**: Scaling will be applied to the **numerical columns** (such as `Tenure Months`, `Monthly Charges`, `Total Charges`, etc.).


In [30]:
df.head()

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,...,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Score,CLTV
0,1,0,0,0,2,1,0,0,2,2,...,0,0,0,1,3,53.85,-0.379687,1,86,-0.673816
1,0,0,0,1,2,1,0,1,0,0,...,0,0,0,1,2,70.70,-0.366848,1,67,-0.955271
2,0,0,0,1,8,1,2,1,0,0,...,2,2,0,1,2,99.65,-0.169434,1,86,0.442061
3,0,0,1,1,28,1,2,1,0,0,...,2,2,0,1,2,104.80,0.487449,1,84,0.249019
4,1,0,0,1,49,1,2,1,0,2,...,2,2,0,1,0,103.70,1.074881,1,89,0.425320


In [31]:
df.to_excel('Telco_customer_churn_processed.xlsx', index=False)

In [18]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Select only numerical columns for scaling
numerical_columns = df.select_dtypes(include=['float64', 'float32', 'int64']).columns

# Apply scaling to the selected numerical columns
df[numerical_columns] = scaler.fit_transform(df[numerical_columns])

# Verify the scaling
df.head()


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,...,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Score,CLTV
0,0.990532,-0.439916,-0.966622,-0.548093,-1.236724,0.327438,-0.991588,-1.183234,1.407321,1.242550,...,-1.113495,-1.121405,-0.828207,0.829798,1.334863,-0.362660,-0.958066,1.663829,1.268402,-0.981675
1,-1.009559,-0.439916,-0.966622,1.824507,-1.236724,0.327438,-0.991588,0.172250,-0.918838,-1.029919,...,-1.113495,-1.121405,-0.828207,0.829798,0.398558,0.197365,-0.938874,1.663829,0.385650,-1.436462
2,-1.009559,-0.439916,-0.966622,1.824507,-0.992402,0.327438,1.117034,0.172250,-0.918838,-1.029919,...,1.146547,1.138411,-0.828207,0.829798,0.398558,1.159546,-0.643789,1.663829,1.268402,0.821409
3,-1.009559,-0.439916,1.034530,1.824507,-0.177995,0.327438,1.117034,0.172250,-0.918838,-1.029919,...,1.146547,1.138411,-0.828207,0.829798,0.398558,1.330711,0.338085,1.663829,1.175481,0.509483
4,0.990532,-0.439916,-0.966622,1.824507,0.677133,0.327438,1.117034,0.172250,-0.918838,1.242550,...,1.146547,1.138411,-0.828207,0.829798,-1.474052,1.294151,1.216150,1.663829,1.407784,0.794358


> Now we have done with the data processing.
on the next files, we will continue with the modeling steps.

+ => [Model Implementation](https://github.com/MOHSIN184/Telecom_Churn_Prediction/blob/main/5.%20Model_Implementation.ipynb)
